# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [6]:
import os
import duckdb
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

assert HF_TOKEN, "HF_TOKEN is missing. Add it in Colab Secrets first."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 = mid-panel month.
# Do not use the _sample table because it is June 2026/final month.
MARCH_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Using March 2026 daily performance partition.")

Connected to FlyRank warehouse.
Using March 2026 daily performance partition.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract answers, in plain words

1. One row in my final modeling frame means one pseudonymized client-content item pair, aggregated from daily Search Console rows.

2. I will use `fact_content_daily_performance`, specifically the March 2026 partition. The raw warehouse grain is `report_date × client_hash_id × content_hash_id`.

3. My feature window is 2026-03-01 through 2026-03-21. My decision moment is the end of 2026-03-21. My outcome window is 2026-03-22 through 2026-03-31.

4. I will predict a proxy label: whether the content’s average daily GSC impressions in the outcome window drop below 80% of its average daily GSC impressions in the feature window. The useful output is a decline-risk score that can rank content for review.

5. I deliberately exclude all post-decision outcome values from the honest feature set, especially future impressions and future-to-history ratios. Those are allowed only for creating the label and the deliberate leakage test.

In [7]:
contract_window = pd.DataFrame(
    [
        {"piece": "feature_start", "value": "2026-03-01"},
        {"piece": "decision_moment", "value": "end of 2026-03-21"},
        {"piece": "outcome_start", "value": "2026-03-22"},
        {"piece": "outcome_end", "value": "2026-03-31"},
        {"piece": "sealed_test_month", "value": "2026-06, not used here"},
    ]
)

display(contract_window)

,piece,value
0,feature_start,2026-03-01
1,decision_moment,end of 2026-03-21
2,outcome_start,2026-03-22
3,outcome_end,2026-03-31
4,sealed_test_month,"2026-06, not used here"


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field buckets

Feature fields, max five:

- `hist_imp_per_day`
- `hist_click_per_day`
- `hist_ctr`
- `hist_avg_position`
- `hist_position_sd`

Label / proxy:

- `is_declining_proxy`

Context fields:

- `client_hash_id`
- `content_hash_id`
- `hist_days`
- `outcome_days`

Excluded fields:

- `label_outcome_imp_per_day`
- `leak_outcome_ratio`
- raw future impressions
- June 2026 sample/final month

`ga4_data_available` is used as an availability filter/check. It is not used as a model feature.

In [8]:
field_contract = pd.DataFrame(
    [
        {"bucket": "feature", "field": "hist_imp_per_day", "reason": "known before decision moment"},
        {"bucket": "feature", "field": "hist_click_per_day", "reason": "known before decision moment"},
        {"bucket": "feature", "field": "hist_ctr", "reason": "computed only from feature-window clicks/impressions"},
        {"bucket": "feature", "field": "hist_avg_position", "reason": "computed only from feature-window GSC position"},
        {"bucket": "feature", "field": "hist_position_sd", "reason": "computed only from feature-window GSC position"},
        {"bucket": "label", "field": "is_declining_proxy", "reason": "computed from future outcome window"},
        {"bucket": "context", "field": "client_hash_id", "reason": "identity/grouping key, not a model feature"},
        {"bucket": "context", "field": "content_hash_id", "reason": "identity/grouping key, not a model feature"},
        {"bucket": "excluded", "field": "label_outcome_imp_per_day", "reason": "future value used only to form label/leak test"},
        {"bucket": "excluded", "field": "leak_outcome_ratio", "reason": "label-derived leakage column"},
    ]
)

display(field_contract)

,bucket,field,reason
0,feature,hist_imp_per_day,known before decision moment
1,feature,hist_click_per_day,known before decision moment
2,feature,hist_ctr,computed only from feature-window clicks/impre...
3,feature,hist_avg_position,computed only from feature-window GSC position
4,feature,hist_position_sd,computed only from feature-window GSC position
5,label,is_declining_proxy,computed from future outcome window
6,context,client_hash_id,"identity/grouping key, not a model feature"
7,context,content_hash_id,"identity/grouping key, not a model feature"
8,excluded,label_outcome_imp_per_day,future value used only to form label/leak test
9,excluded,leak_outcome_ratio,label-derived leakage column


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# VERIFICATION QUERY 1 OF 3: grain check

grain_sql = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {MARCH_DAILY}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 10
"""

grain_check = con.sql(grain_sql).df()
display(grain_check)

if grain_check.empty:
    print("PASS: no duplicate report_date × client_hash_id × content_hash_id rows found.")
else:
    print("CHECK: duplicates appeared. Investigate before continuing.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


PASS: no duplicate report_date × client_hash_id × content_hash_id rows found.


In [10]:
# VERIFICATION QUERY 2 OF 3: row count and date span

count_span_sql = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {MARCH_DAILY}
"""

count_span = con.sql(count_span_sql).df()
display(count_span)

,total_rows,clients,content_items,min_report_date,max_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [11]:
# VERIFICATION QUERY 3 OF 3: availability checked with IS TRUE

availability_sql = f"""
WITH month_rows AS (
    SELECT *
    FROM {MARCH_DAILY}
),
available_rows AS (
    SELECT *
    FROM month_rows
    WHERE ga4_data_available IS TRUE
)
SELECT
    (SELECT COUNT(*) FROM month_rows) AS all_rows,
    (SELECT COUNT(*) FROM available_rows) AS rows_surviving_ga4_available,
    ROUND(
        100.0 * (SELECT COUNT(*) FROM available_rows)
        / NULLIF((SELECT COUNT(*) FROM month_rows), 0),
        2
    ) AS pct_surviving
"""

availability_check = con.sql(availability_sql).df()
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,rows_surviving_ga4_available,pct_surviving
0,9841378,413966,4.21


### Five features and when they are available

- `hist_imp_per_day` — knowable at the decision moment because it uses only GSC impressions from 2026-03-01 to 2026-03-21.
- `hist_click_per_day` — knowable at the decision moment because it uses only GSC clicks from 2026-03-01 to 2026-03-21.
- `hist_ctr` — knowable at the decision moment because it is computed from feature-window clicks divided by feature-window impressions.
- `hist_avg_position` — knowable at the decision moment because it uses only feature-window GSC average position.
- `hist_position_sd` — knowable at the decision moment because it measures only feature-window position variation.

In [12]:
feature_sql = f"""
WITH agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT CASE
            WHEN report_date < DATE '2026-03-22'
            THEN report_date
        END) AS hist_days,

        COUNT(DISTINCT CASE
            WHEN report_date >= DATE '2026-03-22'
            THEN report_date
        END) AS outcome_days,

        SUM(CASE
            WHEN report_date < DATE '2026-03-22'
            THEN gsc_impressions ELSE 0
        END) AS hist_impressions,

        SUM(CASE
            WHEN report_date < DATE '2026-03-22'
            THEN gsc_clicks ELSE 0
        END) AS hist_clicks,

        SUM(CASE
            WHEN report_date >= DATE '2026-03-22'
            THEN gsc_impressions ELSE 0
        END) AS label_outcome_impressions,

        SUM(CASE
            WHEN report_date < DATE '2026-03-22'
            THEN gsc_avg_position * gsc_impressions ELSE 0
        END)
        / NULLIF(
            SUM(CASE
                WHEN report_date < DATE '2026-03-22'
                THEN gsc_impressions ELSE 0
            END),
            0
        ) AS hist_avg_position,

        STDDEV_SAMP(CASE
            WHEN report_date < DATE '2026-03-22'
                 AND gsc_impressions > 0
            THEN gsc_avg_position
        END) AS hist_position_sd

    FROM {MARCH_DAILY}
    WHERE ga4_data_available IS TRUE
    GROUP BY 1, 2
),

framed AS (
    SELECT
        *,
        1.0 * hist_impressions / NULLIF(hist_days, 0) AS hist_imp_per_day,
        1.0 * hist_clicks / NULLIF(hist_days, 0) AS hist_click_per_day,
        1.0 * hist_clicks / NULLIF(hist_impressions, 0) AS hist_ctr,
        1.0 * label_outcome_impressions / NULLIF(outcome_days, 0) AS label_outcome_imp_per_day
    FROM agg
    WHERE hist_days >= 14
      AND outcome_days >= 7
      AND hist_impressions >= 50
)

SELECT
    client_hash_id,
    content_hash_id,
    hist_days,
    outcome_days,

    hist_imp_per_day,
    hist_click_per_day,
    hist_ctr,
    hist_avg_position,
    hist_position_sd,

    label_outcome_imp_per_day,

    CAST(
        label_outcome_imp_per_day < 0.8 * hist_imp_per_day
        AS INTEGER
    ) AS is_declining_proxy

FROM framed
"""

feature_frame = con.sql(feature_sql).df()

honest_features = [
    "hist_imp_per_day",
    "hist_click_per_day",
    "hist_ctr",
    "hist_avg_position",
    "hist_position_sd",
]

display(feature_frame[
    ["client_hash_id", "content_hash_id"] + honest_features + ["is_declining_proxy"]
].head())

print("Rows in feature frame:", len(feature_frame))
print("Number of honest features:", len(honest_features))
print("Duplicate client-content rows:", feature_frame.duplicated(["client_hash_id", "content_hash_id"]).sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,hist_imp_per_day,hist_click_per_day,hist_ctr,hist_avg_position,hist_position_sd,is_declining_proxy
0,client_65de48885f4ef01b,content_e25ea7297a1dffd3,177.235294,1.000000,0.005642,4.144374,0.797423,1
1,client_e547b89c05043229,content_a6eb550e132505fd,555.380952,5.809524,0.010460,5.052388,0.809721,0
2,client_e547b89c05043229,content_cbb44b4a10088d7b,264.578947,2.736842,0.010344,2.871295,0.479089,0
3,client_e547b89c05043229,content_5538d1fc8d19fa4a,673.238095,4.047619,0.006012,3.487976,0.419710,0
4,client_e547b89c05043229,content_cf8a731286a0394b,1280.150000,3.950000,0.003086,2.291489,0.359885,0


Rows in feature frame: 2348
Number of honest features: 5
Duplicate client-content rows: 0


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

model_frame = feature_frame[
    ["client_hash_id", "content_hash_id", "label_outcome_imp_per_day", "is_declining_proxy"] + honest_features
].replace([np.inf, -np.inf], np.nan).dropna()

assert model_frame["is_declining_proxy"].nunique() == 2, "Need both classes for ROC-AUC."

train_idx, test_idx = train_test_split(
    model_frame.index,
    test_size=0.25,
    random_state=42,
    stratify=model_frame["is_declining_proxy"]
)

def quick_auc(df, features):
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=10,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(
        df.loc[train_idx, features],
        df.loc[train_idx, "is_declining_proxy"]
    )

    pred = model.predict_proba(df.loc[test_idx, features])[:, 1]

    return roc_auc_score(
        df.loc[test_idx, "is_declining_proxy"],
        pred
    )

honest_auc = quick_auc(model_frame, honest_features)

leaky_frame = model_frame.copy()
leaky_frame["leak_outcome_ratio"] = (
    leaky_frame["label_outcome_imp_per_day"]
    / leaky_frame["hist_imp_per_day"]
)

leaky_features = honest_features + ["leak_outcome_ratio"]
leaky_auc = quick_auc(leaky_frame, leaky_features)

leaky_frame = leaky_frame.drop(columns=["leak_outcome_ratio"])
final_features_kept = honest_features.copy()

score_report = pd.DataFrame(
    [
        {"model": "honest_features_only", "roc_auc": honest_auc},
        {"model": "with_deliberate_leak", "roc_auc": leaky_auc},
        {"model": "final_kept_score", "roc_auc": honest_auc},
    ]
)

display(score_report)

print("Final honest features kept:", final_features_kept)

,model,roc_auc
0,honest_features_only,0.727365
1,with_deliberate_leak,1.000000
2,final_kept_score,0.727365


Final honest features kept: ['hist_imp_per_day', 'hist_click_per_day', 'hist_ctr', 'hist_avg_position', 'hist_position_sd']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitation

This slice only keeps March 2026 rows where `ga4_data_available IS TRUE` and then requires enough history and outcome days for each client-content pair. That means the frame may underrepresent clients or content with shorter, newer, or less complete tracking history. The result is useful for a first modeling frame, but it should not be treated as a complete view of all clients or all content.

In [14]:
availability_row = availability_check.iloc[0]

print(
    "GA4-available rows surviving:",
    int(availability_row["rows_surviving_ga4_available"]),
    "out of",
    int(availability_row["all_rows"]),
)

print("Percent surviving:", availability_row["pct_surviving"])
print("Final feature-frame rows:", len(feature_frame))

GA4-available rows surviving: 413966 out of 9841378
Percent surviving: 4.21
Final feature-frame rows: 2348


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.